# Ohio Voter Registration Analysis

**Workflow:** Download check → Load → Clean → Enrich → Explore → Export JSON + Excel

Run cells top-to-bottom on first use. After the initial load, you can re-run any individual cell without restarting.

| Cell | Purpose | Re-run cost |
|---|---|---|
| 1 — Imports | Load libraries and logger | Instant |
| 2 — Config | Set county, paths | Instant |
| 3 — Load | Read SWVF files from disk | 10–30 s (county) / 2–4 min (Ohio) |
| 4 — Clean | Parse dates, normalise fields | 5–15 s |
| 5 — Participation | Compute per-voter metrics | 10–30 s |
| 6 — Explore | Ad-hoc queries (scratch space) | Instant |
| 7 — Export JSON | Write web dashboard data | 5–10 s |
| 8 — Export Excel | Write .xlsx workbook | 30–120 s |

In [ ]:
# --- Cell 1: Imports and logging setup ---------------------------------------
# Run once per session. Restart kernel and re-run from here if you edit
# voter_data_cleaner_v2.py — Python's import cache won't pick up changes.

import sys
from pathlib import Path
from datetime import date

import polars as pl
import pandas as pd

from voter_data_cleaner_v2 import (
    setup_logging,
    _timer,
    load_voter_files,
    clean_voter_data,
    identify_election_cols,
    add_voter_participation,
    build_decade_summary,
    build_election_participation,
    build_district_breakdown,
    build_party_crosstabs,
    build_precinct_summary,
    build_city_summary,
    build_city_precinct_summary,
    build_county_summary,
    export_json,
    build_workbook,
    get_source_date,
    OHIO_COUNTIES,
)

log = setup_logging('notebook')
log.info('-' * 50)
log.info('Polars version : %s', pl.__version__)
log.info('Pandas version : %s', pd.__version__)
log.info('Python         : %s', sys.version.split()[0])
log.info('-' * 50)

In [ ]:
# --- Cell 2: Configuration ---------------------------------------------------
# Set COUNTY_NUMBER to a zero-padded string, or None for all of Ohio.
#
#   '57' = Montgomery (Dayton)   '25' = Franklin (Columbus)
#   '18' = Cuyahoga (Cleveland)  '31' = Hamilton (Cincinnati)
#   '77' = Summit (Akron)        '76' = Stark (Canton)

BASE_DIR      = Path().resolve()
TXT_DIR       = BASE_DIR / 'source' / 'State Voter Files'
TXT_FILES     = sorted(TXT_DIR.glob('SWVF_*.txt'))

COUNTY_NUMBER = '57'    # <- change this; or None for all of Ohio
COUNTY_NAME   = OHIO_COUNTIES.get(COUNTY_NUMBER, f'County {COUNTY_NUMBER}') \
                if COUNTY_NUMBER else 'Ohio (Statewide)'

log.info('County        : %s (%s)', COUNTY_NAME, COUNTY_NUMBER or 'all')
log.info('Voter files   : %d found', len(TXT_FILES))
for f in TXT_FILES:
    log.info('  %-25s  %.2f GB', f.name, f.stat().st_size / 1e9)

if not TXT_FILES:
    log.error('No SWVF_*.txt files found in %s', TXT_DIR)
    log.error('Run ohio_voter_pipeline.py first to download and decompress the voter files.')

In [ ]:
# --- Cell 3: Load voter files ------------------------------------------------
# Expected time: single county 10-30 s, all of Ohio 2-4 min.

with _timer(log, 'load voter files'):
    df_raw = load_voter_files(
        txt_files     = TXT_FILES,
        county_number = COUNTY_NUMBER,
        logger        = log,
    )

print(f'\nRows: {len(df_raw):,}   Columns: {len(df_raw.columns)}')
df_raw.head(3)

In [ ]:
# --- Cell 4: Clean and enrich ------------------------------------------------
# Parses dates, derives Decade + Generation cohorts, normalises PARTY_AFFILIATION,
# drops records with invalid birth years.

with _timer(log, 'clean voter data'):
    df = clean_voter_data(df_raw, log)

print(f'\nRecords after cleaning: {len(df):,}')
df.group_by('PARTY_LABEL').len().sort('len', descending=True)

In [ ]:
# --- Cell 5: Participation metrics ------------------------------------------
# Adds Elections_Eligible, Elections_Voted, Turnout_Rate,
# General_Eligible, General_Voted, General_Turnout_Rate, Voter_Frequency.

election_cols = identify_election_cols(df)
log.info('%d election columns detected: %s -> %s',
         len(election_cols), election_cols[0], election_cols[-1])

with _timer(log, 'compute participation metrics'):
    df = add_voter_participation(df, election_cols, log)

df.group_by('Voter_Frequency').len().sort('len', descending=True)

In [ ]:
# --- Cell 6: Explore ---------------------------------------------------------
# Scratch space. Edit and re-run freely.

# Example 1: Top 10 precincts by total registered voters
(
    df.group_by('PRECINCT_NAME')
      .agg([
          pl.len().alias('Total'),
          pl.col('PARTY_LABEL').eq('REP').sum().alias('REP'),
          pl.col('PARTY_LABEL').eq('DEM').sum().alias('DEM'),
          pl.col('PARTY_LABEL').eq('UNC').sum().alias('UNC'),
      ])
      .sort('Total', descending=True)
      .head(10)
)

In [ ]:
# --- Cell 6b: Explore - voter status by congressional district ---------------

(
    df.group_by(['CONGRESSIONAL_DISTRICT', 'VOTER_STATUS'])
      .agg(pl.len().alias('Count'))
      .pivot(on='VOTER_STATUS', index='CONGRESSIONAL_DISTRICT',
             values='Count', aggregate_function='sum')
      .sort('CONGRESSIONAL_DISTRICT')
)

In [ ]:
# --- Cell 6c: Explore - most recent general election turnout by party --------

recent_general = [c for c in election_cols if c.startswith('GENERAL-')][-1]
log.info('Most recent General election column: %s', recent_general)

(
    df.with_columns(
        pl.col(recent_general).is_not_null()
          .and_(pl.col(recent_general).str.strip_chars() != '')
          .alias('voted')
    )
    .group_by('PARTY_LABEL')
    .agg([
        pl.len().alias('Total Registered'),
        pl.col('voted').sum().alias('Voted'),
    ])
    .with_columns(
        (pl.col('Voted') / pl.col('Total Registered') * 100).round(1).alias('Turnout %')
    )
    .sort('Turnout %', descending=True)
)

In [ ]:
# --- Cell 7: Export JSON (web dashboard) ------------------------------------
# Writes JSON files into docs/data/ and updates docs/manifest.json.
# Note: if COUNTY_NUMBER is None (statewide), export_json is skipped with a
# warning — use run_ohio_analysis() from ohio_voter_pipeline.py instead,
# which exports per-county JSON from the statewide DataFrame.

with _timer(log, f'JSON export - {COUNTY_NAME}'):
    export_json(COUNTY_NAME, df, election_cols, log)

print(f'\nDashboard data written to: {BASE_DIR / "docs" / "data"}')
print('Open docs/index.html to view.')

In [ ]:
# --- Cell 8: Export Excel workbook ------------------------------------------
# include_raw=True  -> Voter Data sheet (county runs only; ~300K rows OK)
# include_raw=False -> County Summary sheet (statewide; 7.9M rows > Excel limit)

include_raw = COUNTY_NUMBER is not None

src_date = get_source_date(log)

if COUNTY_NUMBER:
    output_path = BASE_DIR / f'county_{COUNTY_NUMBER}_analysis_src{src_date}.xlsx'
else:
    output_path = BASE_DIR / f'ohio_analysis_src{src_date}.xlsx'

with _timer(log, 'write Excel workbook'):
    build_workbook(
        df            = df,
        election_cols = election_cols,
        output_path   = output_path,
        county_name   = COUNTY_NAME,
        include_raw   = include_raw,
        logger        = log,
    )

print(f'\nWorkbook saved: {output_path}')
print(f'Size: {output_path.stat().st_size / 1e6:.1f} MB')

---
## Reference

### Switching counties
Change `COUNTY_NUMBER` in Cell 2, then re-run Cells 2 -> 3 -> 4 -> 5 -> 7 -> 8.

### Running for all of Ohio
Set `COUNTY_NUMBER = None` in Cell 2, then re-run from Cell 2.
Cell 8 switches to `include_raw=False` automatically.
Cell 7 will skip JSON export with a warning (use ohio_voter_pipeline.py for that).

### Downloading updated voter files
Run `python ohio_voter_pipeline.py` from the terminal, then re-run from Cell 3.

### Ohio county numbers (quick reference)

| # | County | # | County | # | County |
|---|---|---|---|---|---|
| 18 | Cuyahoga (Cleveland) | 25 | Franklin (Columbus) | 31 | Hamilton (Cincinnati) |
| 57 | Montgomery (Dayton) | 76 | Stark (Canton) | 77 | Summit (Akron) |
| 47 | Lorain | 48 | Lucas (Toledo) | 78 | Trumbull (Warren) |